In [1]:
! hostname

g02


In [2]:
! nvidia-smi

Sun May  3 21:53:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3080 Ti     On  |   00000000:86:00.0 Off |                  N/A |
| 30%   18C    P8             23W /  350W |       1MiB /  12288MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# enable autoreload
%load_ext autoreload
%autoreload 2

In [4]:
import os
import sys
import scanpy as sc
import numpy as np
import pandas as pd

import torch
from anndata import AnnData
import anndata
import seaborn as sns
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

In [5]:
plt.rcParams['pdf.fonttype'] = 42
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.set_figure_params(dpi=300,
                     dpi_save=1200,
#                      facecolor='w',
#                      frameon=False, # frameon=True
#                      figsize=(4,4)
                    ) 
%config InlineBackend.figure_format='retina'
%matplotlib inline

scanpy==1.10.3 anndata==0.10.9 umap==0.5.9.post2 numpy==1.26.4 scipy==1.13.1 pandas==2.2.2 scikit-learn==1.6.1 statsmodels==0.14.5 igraph==0.11.9 pynndescent==0.5.13


In [6]:
import sys

repo_path = "/share/home/liangzhongming/phd_code/ModelTest/CellNiche"

if repo_path in sys.path:
    sys.path.remove(repo_path)

sys.path.insert(0, repo_path)

for m in list(sys.modules.keys()):
    if m == "cellniche" or m.startswith("cellniche."):
        del sys.modules[m]

import cellniche as cn

In [7]:
! cat ../configs/sim.yaml

# ============================================================
#  CellNiche multi-slice training config
# ============================================================

# ------------------------------------------------------------
#  DATA & PRE-PROCESSING
# ------------------------------------------------------------
data_path: "/share/home/liangzhongming/phd_code/st_data/SpaLPdata/640,000-simulated-data/"
dataset: "sim"

# Cell identity label in adata.obs.
# Required when embedding_type is "pheno" or "pheno_expr".
# Not required when embedding_type is "expr".
phenoLabels: null

# Ground-truth niche / region label in adata.obs.
# Optional. Set to null if no ground-truth annotation is available.
nicheLabels: 'ann_level_3'

# Input feature mode:
#   "pheno"      : use one-hot encoded phenoLabels as input features
#   "expr"       : use adata.X as input features
#   "pheno_expr" : use phenoLabels as model input and adata.X for expression-based positive-pair construction
#   "embedding"  :

In [8]:
simulation = cn.cli(["--config", "../configs/sim.yaml"])

21:53:39 INFO: Seed: 110
21:53:39 INFO: Using device: cuda


extracting highly variable genes
--> added
    'highly_variable', boolean vector (adata.var)
    'highly_variable_rank', float vector (adata.var)
    'means', float vector (adata.var)
    'variances', float vector (adata.var)
    'variances_norm', float vector (adata.var)
normalizing counts per cell
    finished (0:00:02)


21:54:47 INFO: Average number of neighbors per node: 6.0
21:54:48 INFO: Loaded sim: 640000 nodes, 3685532 edges, 256 expression features
21:54:48 INFO: loading_time: 68.92s
21:54:50 INFO: Epoch 1 Step 0001 contrast_loss=8.3224, recon_loss=0.0000
21:54:50 INFO: Epoch 1 Step 0002 contrast_loss=8.1320, recon_loss=0.0000
21:54:50 INFO: Epoch 1 Step 0003 contrast_loss=8.0299, recon_loss=0.0000
21:54:50 INFO: Epoch 1 Step 0004 contrast_loss=7.9921, recon_loss=0.0000
21:54:50 INFO: Epoch 1 Step 0005 contrast_loss=8.0330, recon_loss=0.0000
21:54:50 INFO: Epoch 1 Step 0006 contrast_loss=7.9673, recon_loss=0.0000
21:54:50 INFO: Epoch 1 Step 0007 contrast_loss=8.0013, recon_loss=0.0000
21:54:50 INFO: Epoch 1 Step 0008 contrast_loss=7.9603, recon_loss=0.0000
21:54:50 INFO: Epoch 1 Step 0009 contrast_loss=7.9174, recon_loss=0.0000
21:54:50 INFO: Epoch 1 Step 0010 contrast_loss=7.9661, recon_loss=0.0000
21:54:50 INFO: Epoch 1 Step 0011 contrast_loss=7.9358, recon_loss=0.0000
21:54:50 INFO: Epoch 1 S

21:54:54 INFO: Epoch 1 Step 0111 contrast_loss=7.7637, recon_loss=0.0000
21:54:54 INFO: Epoch 1 Step 0112 contrast_loss=7.7663, recon_loss=0.0000
21:54:54 INFO: Epoch 1 Step 0113 contrast_loss=7.7616, recon_loss=0.0000
21:54:54 INFO: Epoch 1 Step 0114 contrast_loss=7.7645, recon_loss=0.0000
21:54:54 INFO: Epoch 1 Step 0115 contrast_loss=7.7229, recon_loss=0.0000
21:54:54 INFO: Epoch 1 Step 0116 contrast_loss=7.7617, recon_loss=0.0000
21:54:54 INFO: Epoch 1 Step 0117 contrast_loss=7.7573, recon_loss=0.0000
21:54:54 INFO: Epoch 1 Step 0118 contrast_loss=7.7751, recon_loss=0.0000
21:54:54 INFO: Epoch 1 Step 0119 contrast_loss=7.7370, recon_loss=0.0000
21:54:54 INFO: Epoch 1 Step 0120 contrast_loss=7.7291, recon_loss=0.0000
21:54:54 INFO: Epoch 1 Step 0121 contrast_loss=7.7484, recon_loss=0.0000
21:54:54 INFO: Epoch 1 Step 0122 contrast_loss=7.7584, recon_loss=0.0000
21:54:54 INFO: Epoch 1 Step 0123 contrast_loss=7.7609, recon_loss=0.0000
21:54:54 INFO: Epoch 1 Step 0124 contrast_loss=7.75

21:54:58 INFO: Epoch 1 Step 0224 contrast_loss=7.6905, recon_loss=0.0000
21:54:58 INFO: Epoch 1 Step 0225 contrast_loss=7.6644, recon_loss=0.0000
21:54:58 INFO: Epoch 1 Step 0226 contrast_loss=7.7143, recon_loss=0.0000
21:54:58 INFO: Epoch 1 Step 0227 contrast_loss=7.6945, recon_loss=0.0000
21:54:58 INFO: Epoch 1 Step 0228 contrast_loss=7.6643, recon_loss=0.0000
21:54:58 INFO: Epoch 1 Step 0229 contrast_loss=7.6572, recon_loss=0.0000
21:54:58 INFO: Epoch 1 Step 0230 contrast_loss=7.6668, recon_loss=0.0000
21:54:58 INFO: Epoch 1 Step 0231 contrast_loss=7.6666, recon_loss=0.0000
21:54:58 INFO: Epoch 1 Step 0232 contrast_loss=7.6762, recon_loss=0.0000
21:54:58 INFO: Epoch 1 Step 0233 contrast_loss=7.6658, recon_loss=0.0000
21:54:58 INFO: Epoch 1 Step 0234 contrast_loss=7.7038, recon_loss=0.0000
21:54:58 INFO: Epoch 1 Step 0235 contrast_loss=7.6890, recon_loss=0.0000
21:54:58 INFO: Epoch 1 Step 0236 contrast_loss=7.6455, recon_loss=0.0000
21:54:58 INFO: Epoch 1 Step 0237 contrast_loss=7.70

...

21:56:06 INFO: Epoch 2 Step 2032 contrast_loss=4.5449, recon_loss=0.0000
21:56:06 INFO: Epoch 2 Step 2033 contrast_loss=4.4687, recon_loss=0.0000
21:56:06 INFO: Epoch 2 Step 2034 contrast_loss=4.5261, recon_loss=0.0000
21:56:06 INFO: Epoch 2 Step 2035 contrast_loss=4.5357, recon_loss=0.0000
21:56:06 INFO: Epoch 2 Step 2036 contrast_loss=4.5040, recon_loss=0.0000
21:56:06 INFO: Epoch 2 Step 2037 contrast_loss=4.5441, recon_loss=0.0000
21:56:06 INFO: Epoch 2 Step 2038 contrast_loss=4.4997, recon_loss=0.0000
21:56:06 INFO: Epoch 2 Step 2039 contrast_loss=4.5249, recon_loss=0.0000
21:56:06 INFO: Epoch 2 Step 2040 contrast_loss=4.5514, recon_loss=0.0000
21:56:06 INFO: Epoch 2 Step 2041 contrast_loss=4.4991, recon_loss=0.0000
21:56:06 INFO: Epoch 2 Step 2042 contrast_loss=4.4551, recon_loss=0.0000
21:56:06 INFO: Epoch 2 Step 2043 contrast_loss=4.5101, recon_loss=0.0000
21:56:06 INFO: Epoch 2 Step 2044 contrast_loss=4.5237, recon_loss=0.0000
21:56:06 INFO: Epoch 2 Step 2045 contrast_loss=4.46

21:56:10 INFO: Epoch 2 Step 2145 contrast_loss=4.4463, recon_loss=0.0000
21:56:10 INFO: Epoch 2 Step 2146 contrast_loss=4.4866, recon_loss=0.0000
21:56:10 INFO: Epoch 2 Step 2147 contrast_loss=4.4504, recon_loss=0.0000
21:56:10 INFO: Epoch 2 Step 2148 contrast_loss=4.5497, recon_loss=0.0000
21:56:10 INFO: Epoch 2 Step 2149 contrast_loss=4.4146, recon_loss=0.0000
21:56:10 INFO: Epoch 2 Step 2150 contrast_loss=4.5395, recon_loss=0.0000
21:56:10 INFO: Epoch 2 Step 2151 contrast_loss=4.4452, recon_loss=0.0000
21:56:10 INFO: Epoch 2 Step 2152 contrast_loss=4.4385, recon_loss=0.0000
21:56:10 INFO: Epoch 2 Step 2153 contrast_loss=4.4903, recon_loss=0.0000
21:56:10 INFO: Epoch 2 Step 2154 contrast_loss=4.4975, recon_loss=0.0000
21:56:10 INFO: Epoch 2 Step 2155 contrast_loss=4.4249, recon_loss=0.0000
21:56:10 INFO: Epoch 2 Step 2156 contrast_loss=4.4503, recon_loss=0.0000
21:56:10 INFO: Epoch 2 Step 2157 contrast_loss=4.4569, recon_loss=0.0000
21:56:10 INFO: Epoch 2 Step 2158 contrast_loss=4.49

21:56:14 INFO: Epoch 2 Step 2258 contrast_loss=4.4437, recon_loss=0.0000
21:56:14 INFO: Epoch 2 Step 2259 contrast_loss=4.4873, recon_loss=0.0000
21:56:14 INFO: Epoch 2 Step 2260 contrast_loss=4.4336, recon_loss=0.0000
21:56:14 INFO: Epoch 2 Step 2261 contrast_loss=4.4587, recon_loss=0.0000
21:56:14 INFO: Epoch 2 Step 2262 contrast_loss=4.4471, recon_loss=0.0000
21:56:14 INFO: Epoch 2 Step 2263 contrast_loss=4.4258, recon_loss=0.0000
21:56:14 INFO: Epoch 2 Step 2264 contrast_loss=4.4800, recon_loss=0.0000
21:56:14 INFO: Epoch 2 Step 2265 contrast_loss=4.3958, recon_loss=0.0000
21:56:14 INFO: Epoch 2 Step 2266 contrast_loss=4.4383, recon_loss=0.0000
21:56:14 INFO: Epoch 2 Step 2267 contrast_loss=4.4520, recon_loss=0.0000
21:56:14 INFO: Epoch 2 Step 2268 contrast_loss=4.4365, recon_loss=0.0000
21:56:14 INFO: Epoch 2 Step 2269 contrast_loss=4.4527, recon_loss=0.0000
21:56:14 INFO: Epoch 2 Step 2270 contrast_loss=4.4508, recon_loss=0.0000
21:56:14 INFO: Epoch 2 Step 2271 contrast_loss=4.43

21:56:18 INFO: Epoch 2 Step 2371 contrast_loss=4.3963, recon_loss=0.0000
21:56:18 INFO: Epoch 2 Step 2372 contrast_loss=4.4228, recon_loss=0.0000
21:56:18 INFO: Epoch 2 Step 2373 contrast_loss=4.4261, recon_loss=0.0000
21:56:18 INFO: Epoch 2 Step 2374 contrast_loss=4.4362, recon_loss=0.0000
21:56:18 INFO: Epoch 2 Step 2375 contrast_loss=4.3897, recon_loss=0.0000
21:56:18 INFO: Epoch 2 Step 2376 contrast_loss=4.4264, recon_loss=0.0000
21:56:18 INFO: Epoch 2 Step 2377 contrast_loss=4.4118, recon_loss=0.0000
21:56:18 INFO: Epoch 2 Step 2378 contrast_loss=4.3927, recon_loss=0.0000
21:56:18 INFO: Epoch 2 Step 2379 contrast_loss=4.4272, recon_loss=0.0000
21:56:19 INFO: Epoch 2 Step 2380 contrast_loss=4.4462, recon_loss=0.0000
21:56:19 INFO: Epoch 2 Step 2381 contrast_loss=4.4729, recon_loss=0.0000
21:56:19 INFO: Epoch 2 Step 2382 contrast_loss=4.4954, recon_loss=0.0000
21:56:19 INFO: Epoch 2 Step 2383 contrast_loss=4.3842, recon_loss=0.0000
21:56:19 INFO: Epoch 2 Step 2384 contrast_loss=4.43

21:56:22 INFO: Epoch 2 Step 2484 contrast_loss=4.4324, recon_loss=0.0000
21:56:22 INFO: Epoch 2 Step 2485 contrast_loss=4.4310, recon_loss=0.0000
21:56:22 INFO: Epoch 2 Step 2486 contrast_loss=4.3847, recon_loss=0.0000
21:56:22 INFO: Epoch 2 Step 2487 contrast_loss=4.4157, recon_loss=0.0000
21:56:22 INFO: Epoch 2 Step 2488 contrast_loss=4.4319, recon_loss=0.0000
21:56:22 INFO: Epoch 2 Step 2489 contrast_loss=4.4163, recon_loss=0.0000
21:56:22 INFO: Epoch 2 Step 2490 contrast_loss=4.3977, recon_loss=0.0000
21:56:23 INFO: Epoch 2 Step 2491 contrast_loss=4.4084, recon_loss=0.0000
21:56:23 INFO: Epoch 2 Step 2492 contrast_loss=4.4211, recon_loss=0.0000
21:56:23 INFO: Epoch 2 Step 2493 contrast_loss=4.4008, recon_loss=0.0000
21:56:23 INFO: Epoch 2 Step 2494 contrast_loss=4.3725, recon_loss=0.0000
21:56:23 INFO: Epoch 2 Step 2495 contrast_loss=4.3833, recon_loss=0.0000
21:56:23 INFO: Epoch 2 Step 2496 contrast_loss=4.4421, recon_loss=0.0000
21:56:23 INFO: Epoch 2 Step 2497 contrast_loss=4.42

In [9]:
simulation

AnnData object with n_obs × n_vars = 640000 × 31493
    obs: 'ann_level_3', 'kmeans'
    var: 'feature_is_filtered', 'original_gene_symbols', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type', 'n_cells'
    uns: 'batch_condition', 'citation', 'default_embedding', 'schema_reference', 'schema_version', 'title'
    obsm: 'X_scanvi_emb', 'X_umap', 'spatial', 'CellNiche'
    layers: 'soupX'
    obsp: 'connectivities', 'distances'

In [10]:
sim = cn.cli(["--config", "../configs/sim.yaml"])

21:58:53 INFO: Seed: 3178
21:58:53 INFO: Using device: cuda


extracting highly variable genes
--> added
    'highly_variable', boolean vector (adata.var)
    'highly_variable_rank', float vector (adata.var)
    'means', float vector (adata.var)
    'variances', float vector (adata.var)
    'variances_norm', float vector (adata.var)
normalizing counts per cell
    finished (0:00:02)


22:00:01 INFO: Average number of neighbors per node: 6.0
22:00:01 INFO: Loaded sim: 640000 nodes, 3685532 edges, 256 expression features
22:00:01 INFO: loading_time: 68.16s
22:00:02 INFO: Epoch 1 Step 0001 contrast_loss=8.3140, recon_loss=0.0000
22:00:02 INFO: Epoch 1 Step 0002 contrast_loss=8.1121, recon_loss=0.0000
22:00:02 INFO: Epoch 1 Step 0003 contrast_loss=8.0393, recon_loss=0.0000
22:00:02 INFO: Epoch 1 Step 0004 contrast_loss=8.0108, recon_loss=0.0000
22:00:02 INFO: Epoch 1 Step 0005 contrast_loss=8.0002, recon_loss=0.0000
22:00:02 INFO: Epoch 1 Step 0006 contrast_loss=7.9830, recon_loss=0.0000
22:00:02 INFO: Epoch 1 Step 0007 contrast_loss=7.9775, recon_loss=0.0000
22:00:02 INFO: Epoch 1 Step 0008 contrast_loss=7.9346, recon_loss=0.0000
22:00:03 INFO: Epoch 1 Step 0009 contrast_loss=7.9769, recon_loss=0.0000
22:00:03 INFO: Epoch 1 Step 0010 contrast_loss=7.9083, recon_loss=0.0000
22:00:03 INFO: Epoch 1 Step 0011 contrast_loss=7.9624, recon_loss=0.0000
22:00:03 INFO: Epoch 1 S

22:00:06 INFO: Epoch 1 Step 0111 contrast_loss=7.7300, recon_loss=0.0000
22:00:06 INFO: Epoch 1 Step 0112 contrast_loss=7.7817, recon_loss=0.0000
22:00:07 INFO: Epoch 1 Step 0113 contrast_loss=7.7628, recon_loss=0.0000
22:00:07 INFO: Epoch 1 Step 0114 contrast_loss=7.7485, recon_loss=0.0000
22:00:07 INFO: Epoch 1 Step 0115 contrast_loss=7.7555, recon_loss=0.0000
22:00:07 INFO: Epoch 1 Step 0116 contrast_loss=7.7660, recon_loss=0.0000
22:00:07 INFO: Epoch 1 Step 0117 contrast_loss=7.7829, recon_loss=0.0000
22:00:07 INFO: Epoch 1 Step 0118 contrast_loss=7.7712, recon_loss=0.0000
22:00:07 INFO: Epoch 1 Step 0119 contrast_loss=7.7714, recon_loss=0.0000
22:00:07 INFO: Epoch 1 Step 0120 contrast_loss=7.7650, recon_loss=0.0000
22:00:07 INFO: Epoch 1 Step 0121 contrast_loss=7.7526, recon_loss=0.0000
22:00:07 INFO: Epoch 1 Step 0122 contrast_loss=7.7482, recon_loss=0.0000
22:00:07 INFO: Epoch 1 Step 0123 contrast_loss=7.7670, recon_loss=0.0000
22:00:07 INFO: Epoch 1 Step 0124 contrast_loss=7.77

22:00:11 INFO: Epoch 1 Step 0224 contrast_loss=7.6753, recon_loss=0.0000
22:00:11 INFO: Epoch 1 Step 0225 contrast_loss=7.6688, recon_loss=0.0000
22:00:11 INFO: Epoch 1 Step 0226 contrast_loss=7.6806, recon_loss=0.0000
22:00:11 INFO: Epoch 1 Step 0227 contrast_loss=7.6975, recon_loss=0.0000
22:00:11 INFO: Epoch 1 Step 0228 contrast_loss=7.6636, recon_loss=0.0000
22:00:11 INFO: Epoch 1 Step 0229 contrast_loss=7.6875, recon_loss=0.0000
22:00:11 INFO: Epoch 1 Step 0230 contrast_loss=7.6933, recon_loss=0.0000
22:00:11 INFO: Epoch 1 Step 0231 contrast_loss=7.6940, recon_loss=0.0000
22:00:11 INFO: Epoch 1 Step 0232 contrast_loss=7.6694, recon_loss=0.0000
22:00:11 INFO: Epoch 1 Step 0233 contrast_loss=7.6764, recon_loss=0.0000
22:00:11 INFO: Epoch 1 Step 0234 contrast_loss=7.6648, recon_loss=0.0000
22:00:11 INFO: Epoch 1 Step 0235 contrast_loss=7.6438, recon_loss=0.0000
22:00:11 INFO: Epoch 1 Step 0236 contrast_loss=7.6714, recon_loss=0.0000
22:00:11 INFO: Epoch 1 Step 0237 contrast_loss=7.67

...

22:01:21 INFO: Epoch 2 Step 2032 contrast_loss=4.5251, recon_loss=0.0000
22:01:21 INFO: Epoch 2 Step 2033 contrast_loss=4.5207, recon_loss=0.0000
22:01:21 INFO: Epoch 2 Step 2034 contrast_loss=4.5812, recon_loss=0.0000
22:01:21 INFO: Epoch 2 Step 2035 contrast_loss=4.5040, recon_loss=0.0000
22:01:21 INFO: Epoch 2 Step 2036 contrast_loss=4.4716, recon_loss=0.0000
22:01:21 INFO: Epoch 2 Step 2037 contrast_loss=4.5265, recon_loss=0.0000
22:01:21 INFO: Epoch 2 Step 2038 contrast_loss=4.5077, recon_loss=0.0000
22:01:21 INFO: Epoch 2 Step 2039 contrast_loss=4.5784, recon_loss=0.0000
22:01:21 INFO: Epoch 2 Step 2040 contrast_loss=4.5149, recon_loss=0.0000
22:01:21 INFO: Epoch 2 Step 2041 contrast_loss=4.5138, recon_loss=0.0000
22:01:21 INFO: Epoch 2 Step 2042 contrast_loss=4.5406, recon_loss=0.0000
22:01:21 INFO: Epoch 2 Step 2043 contrast_loss=4.5524, recon_loss=0.0000
22:01:22 INFO: Epoch 2 Step 2044 contrast_loss=4.5761, recon_loss=0.0000
22:01:22 INFO: Epoch 2 Step 2045 contrast_loss=4.52

22:01:25 INFO: Epoch 2 Step 2145 contrast_loss=4.4893, recon_loss=0.0000
22:01:25 INFO: Epoch 2 Step 2146 contrast_loss=4.4404, recon_loss=0.0000
22:01:25 INFO: Epoch 2 Step 2147 contrast_loss=4.4639, recon_loss=0.0000
22:01:26 INFO: Epoch 2 Step 2148 contrast_loss=4.4541, recon_loss=0.0000
22:01:26 INFO: Epoch 2 Step 2149 contrast_loss=4.5081, recon_loss=0.0000
22:01:26 INFO: Epoch 2 Step 2150 contrast_loss=4.4496, recon_loss=0.0000
22:01:26 INFO: Epoch 2 Step 2151 contrast_loss=4.4618, recon_loss=0.0000
22:01:26 INFO: Epoch 2 Step 2152 contrast_loss=4.4738, recon_loss=0.0000
22:01:26 INFO: Epoch 2 Step 2153 contrast_loss=4.4497, recon_loss=0.0000
22:01:26 INFO: Epoch 2 Step 2154 contrast_loss=4.4535, recon_loss=0.0000
22:01:26 INFO: Epoch 2 Step 2155 contrast_loss=4.4780, recon_loss=0.0000
22:01:26 INFO: Epoch 2 Step 2156 contrast_loss=4.4664, recon_loss=0.0000
22:01:26 INFO: Epoch 2 Step 2157 contrast_loss=4.4516, recon_loss=0.0000
22:01:26 INFO: Epoch 2 Step 2158 contrast_loss=4.45

22:01:30 INFO: Epoch 2 Step 2258 contrast_loss=4.4248, recon_loss=0.0000
22:01:30 INFO: Epoch 2 Step 2259 contrast_loss=4.3873, recon_loss=0.0000
22:01:30 INFO: Epoch 2 Step 2260 contrast_loss=4.4166, recon_loss=0.0000
22:01:30 INFO: Epoch 2 Step 2261 contrast_loss=4.4767, recon_loss=0.0000
22:01:30 INFO: Epoch 2 Step 2262 contrast_loss=4.4288, recon_loss=0.0000
22:01:30 INFO: Epoch 2 Step 2263 contrast_loss=4.4279, recon_loss=0.0000
22:01:30 INFO: Epoch 2 Step 2264 contrast_loss=4.4800, recon_loss=0.0000
22:01:30 INFO: Epoch 2 Step 2265 contrast_loss=4.4020, recon_loss=0.0000
22:01:30 INFO: Epoch 2 Step 2266 contrast_loss=4.4806, recon_loss=0.0000
22:01:30 INFO: Epoch 2 Step 2267 contrast_loss=4.4392, recon_loss=0.0000
22:01:30 INFO: Epoch 2 Step 2268 contrast_loss=4.3973, recon_loss=0.0000
22:01:30 INFO: Epoch 2 Step 2269 contrast_loss=4.4624, recon_loss=0.0000
22:01:30 INFO: Epoch 2 Step 2270 contrast_loss=4.4257, recon_loss=0.0000
22:01:30 INFO: Epoch 2 Step 2271 contrast_loss=4.45

22:01:34 INFO: Epoch 2 Step 2371 contrast_loss=4.4407, recon_loss=0.0000
22:01:34 INFO: Epoch 2 Step 2372 contrast_loss=4.3877, recon_loss=0.0000
22:01:34 INFO: Epoch 2 Step 2373 contrast_loss=4.4333, recon_loss=0.0000
22:01:34 INFO: Epoch 2 Step 2374 contrast_loss=4.4115, recon_loss=0.0000
22:01:34 INFO: Epoch 2 Step 2375 contrast_loss=4.4224, recon_loss=0.0000
22:01:34 INFO: Epoch 2 Step 2376 contrast_loss=4.4182, recon_loss=0.0000
22:01:34 INFO: Epoch 2 Step 2377 contrast_loss=4.4198, recon_loss=0.0000
22:01:34 INFO: Epoch 2 Step 2378 contrast_loss=4.3977, recon_loss=0.0000
22:01:34 INFO: Epoch 2 Step 2379 contrast_loss=4.3958, recon_loss=0.0000
22:01:34 INFO: Epoch 2 Step 2380 contrast_loss=4.3993, recon_loss=0.0000
22:01:34 INFO: Epoch 2 Step 2381 contrast_loss=4.4226, recon_loss=0.0000
22:01:34 INFO: Epoch 2 Step 2382 contrast_loss=4.4713, recon_loss=0.0000
22:01:34 INFO: Epoch 2 Step 2383 contrast_loss=4.4071, recon_loss=0.0000
22:01:34 INFO: Epoch 2 Step 2384 contrast_loss=4.43

22:01:38 INFO: Epoch 2 Step 2484 contrast_loss=4.3653, recon_loss=0.0000
22:01:38 INFO: Epoch 2 Step 2485 contrast_loss=4.3905, recon_loss=0.0000
22:01:38 INFO: Epoch 2 Step 2486 contrast_loss=4.3707, recon_loss=0.0000
22:01:38 INFO: Epoch 2 Step 2487 contrast_loss=4.4289, recon_loss=0.0000
22:01:38 INFO: Epoch 2 Step 2488 contrast_loss=4.3850, recon_loss=0.0000
22:01:38 INFO: Epoch 2 Step 2489 contrast_loss=4.4102, recon_loss=0.0000
22:01:38 INFO: Epoch 2 Step 2490 contrast_loss=4.4088, recon_loss=0.0000
22:01:39 INFO: Epoch 2 Step 2491 contrast_loss=4.4402, recon_loss=0.0000
22:01:39 INFO: Epoch 2 Step 2492 contrast_loss=4.3726, recon_loss=0.0000
22:01:39 INFO: Epoch 2 Step 2493 contrast_loss=4.4038, recon_loss=0.0000
22:01:39 INFO: Epoch 2 Step 2494 contrast_loss=4.3891, recon_loss=0.0000
22:01:39 INFO: Epoch 2 Step 2495 contrast_loss=4.4311, recon_loss=0.0000
22:01:39 INFO: Epoch 2 Step 2496 contrast_loss=4.4294, recon_loss=0.0000
22:01:39 INFO: Epoch 2 Step 2497 contrast_loss=4.44

In [11]:
sim = cn.cli(["--config", "../configs/sim.yaml"])

22:06:47 INFO: Seed: 3192
22:06:47 INFO: Using device: cuda


extracting highly variable genes
--> added
    'highly_variable', boolean vector (adata.var)
    'highly_variable_rank', float vector (adata.var)
    'means', float vector (adata.var)
    'variances', float vector (adata.var)
    'variances_norm', float vector (adata.var)
normalizing counts per cell
    finished (0:00:02)


22:07:55 INFO: Average number of neighbors per node: 6.0
22:07:55 INFO: Loaded sim: 640000 nodes, 3685532 edges, 256 expression features
22:07:55 INFO: loading_time: 68.13s
22:07:56 INFO: Epoch 1 Step 0001 contrast_loss=8.2957, recon_loss=0.0000
22:07:56 INFO: Epoch 1 Step 0002 contrast_loss=8.1178, recon_loss=0.0000
22:07:56 INFO: Epoch 1 Step 0003 contrast_loss=8.0537, recon_loss=0.0000
22:07:56 INFO: Epoch 1 Step 0004 contrast_loss=7.9815, recon_loss=0.0000
22:07:56 INFO: Epoch 1 Step 0005 contrast_loss=7.9904, recon_loss=0.0000
22:07:56 INFO: Epoch 1 Step 0006 contrast_loss=7.9787, recon_loss=0.0000
22:07:56 INFO: Epoch 1 Step 0007 contrast_loss=7.9618, recon_loss=0.0000
22:07:56 INFO: Epoch 1 Step 0008 contrast_loss=7.9783, recon_loss=0.0000
22:07:57 INFO: Epoch 1 Step 0009 contrast_loss=7.9464, recon_loss=0.0000
22:07:57 INFO: Epoch 1 Step 0010 contrast_loss=7.9181, recon_loss=0.0000
22:07:57 INFO: Epoch 1 Step 0011 contrast_loss=7.9694, recon_loss=0.0000
22:07:57 INFO: Epoch 1 S

22:08:00 INFO: Epoch 1 Step 0111 contrast_loss=7.7639, recon_loss=0.0000
22:08:00 INFO: Epoch 1 Step 0112 contrast_loss=7.7470, recon_loss=0.0000
22:08:01 INFO: Epoch 1 Step 0113 contrast_loss=7.8030, recon_loss=0.0000
22:08:01 INFO: Epoch 1 Step 0114 contrast_loss=7.7631, recon_loss=0.0000
22:08:01 INFO: Epoch 1 Step 0115 contrast_loss=7.7228, recon_loss=0.0000
22:08:01 INFO: Epoch 1 Step 0116 contrast_loss=7.7407, recon_loss=0.0000
22:08:01 INFO: Epoch 1 Step 0117 contrast_loss=7.7867, recon_loss=0.0000
22:08:01 INFO: Epoch 1 Step 0118 contrast_loss=7.7577, recon_loss=0.0000
22:08:01 INFO: Epoch 1 Step 0119 contrast_loss=7.7458, recon_loss=0.0000
22:08:01 INFO: Epoch 1 Step 0120 contrast_loss=7.7436, recon_loss=0.0000
22:08:01 INFO: Epoch 1 Step 0121 contrast_loss=7.7498, recon_loss=0.0000
22:08:01 INFO: Epoch 1 Step 0122 contrast_loss=7.7409, recon_loss=0.0000
22:08:01 INFO: Epoch 1 Step 0123 contrast_loss=7.7648, recon_loss=0.0000
22:08:01 INFO: Epoch 1 Step 0124 contrast_loss=7.76

22:08:05 INFO: Epoch 1 Step 0224 contrast_loss=7.6866, recon_loss=0.0000
22:08:05 INFO: Epoch 1 Step 0225 contrast_loss=7.6873, recon_loss=0.0000
22:08:05 INFO: Epoch 1 Step 0226 contrast_loss=7.6793, recon_loss=0.0000
22:08:05 INFO: Epoch 1 Step 0227 contrast_loss=7.6956, recon_loss=0.0000
22:08:05 INFO: Epoch 1 Step 0228 contrast_loss=7.6704, recon_loss=0.0000
22:08:05 INFO: Epoch 1 Step 0229 contrast_loss=7.6695, recon_loss=0.0000
22:08:05 INFO: Epoch 1 Step 0230 contrast_loss=7.7072, recon_loss=0.0000
22:08:05 INFO: Epoch 1 Step 0231 contrast_loss=7.6700, recon_loss=0.0000
22:08:05 INFO: Epoch 1 Step 0232 contrast_loss=7.6686, recon_loss=0.0000
22:08:05 INFO: Epoch 1 Step 0233 contrast_loss=7.6835, recon_loss=0.0000
22:08:05 INFO: Epoch 1 Step 0234 contrast_loss=7.6775, recon_loss=0.0000
22:08:05 INFO: Epoch 1 Step 0235 contrast_loss=7.6660, recon_loss=0.0000
22:08:05 INFO: Epoch 1 Step 0236 contrast_loss=7.6884, recon_loss=0.0000
22:08:05 INFO: Epoch 1 Step 0237 contrast_loss=7.66

...

22:09:16 INFO: Epoch 2 Step 2032 contrast_loss=4.5495, recon_loss=0.0000
22:09:16 INFO: Epoch 2 Step 2033 contrast_loss=4.5741, recon_loss=0.0000
22:09:16 INFO: Epoch 2 Step 2034 contrast_loss=4.5599, recon_loss=0.0000
22:09:16 INFO: Epoch 2 Step 2035 contrast_loss=4.5339, recon_loss=0.0000
22:09:16 INFO: Epoch 2 Step 2036 contrast_loss=4.5163, recon_loss=0.0000
22:09:16 INFO: Epoch 2 Step 2037 contrast_loss=4.4894, recon_loss=0.0000
22:09:16 INFO: Epoch 2 Step 2038 contrast_loss=4.5843, recon_loss=0.0000
22:09:16 INFO: Epoch 2 Step 2039 contrast_loss=4.5270, recon_loss=0.0000
22:09:16 INFO: Epoch 2 Step 2040 contrast_loss=4.5375, recon_loss=0.0000
22:09:16 INFO: Epoch 2 Step 2041 contrast_loss=4.4987, recon_loss=0.0000
22:09:16 INFO: Epoch 2 Step 2042 contrast_loss=4.5367, recon_loss=0.0000
22:09:16 INFO: Epoch 2 Step 2043 contrast_loss=4.5423, recon_loss=0.0000
22:09:16 INFO: Epoch 2 Step 2044 contrast_loss=4.4576, recon_loss=0.0000
22:09:16 INFO: Epoch 2 Step 2045 contrast_loss=4.56

22:09:20 INFO: Epoch 2 Step 2145 contrast_loss=4.4241, recon_loss=0.0000
22:09:20 INFO: Epoch 2 Step 2146 contrast_loss=4.4516, recon_loss=0.0000
22:09:20 INFO: Epoch 2 Step 2147 contrast_loss=4.4528, recon_loss=0.0000
22:09:20 INFO: Epoch 2 Step 2148 contrast_loss=4.4818, recon_loss=0.0000
22:09:20 INFO: Epoch 2 Step 2149 contrast_loss=4.4969, recon_loss=0.0000
22:09:20 INFO: Epoch 2 Step 2150 contrast_loss=4.4927, recon_loss=0.0000
22:09:21 INFO: Epoch 2 Step 2151 contrast_loss=4.4819, recon_loss=0.0000
22:09:21 INFO: Epoch 2 Step 2152 contrast_loss=4.4428, recon_loss=0.0000
22:09:21 INFO: Epoch 2 Step 2153 contrast_loss=4.4394, recon_loss=0.0000
22:09:21 INFO: Epoch 2 Step 2154 contrast_loss=4.4291, recon_loss=0.0000
22:09:21 INFO: Epoch 2 Step 2155 contrast_loss=4.4648, recon_loss=0.0000
22:09:21 INFO: Epoch 2 Step 2156 contrast_loss=4.4464, recon_loss=0.0000
22:09:21 INFO: Epoch 2 Step 2157 contrast_loss=4.4132, recon_loss=0.0000
22:09:21 INFO: Epoch 2 Step 2158 contrast_loss=4.48

22:09:25 INFO: Epoch 2 Step 2258 contrast_loss=4.4829, recon_loss=0.0000
22:09:25 INFO: Epoch 2 Step 2259 contrast_loss=4.4103, recon_loss=0.0000
22:09:25 INFO: Epoch 2 Step 2260 contrast_loss=4.4684, recon_loss=0.0000
22:09:25 INFO: Epoch 2 Step 2261 contrast_loss=4.4378, recon_loss=0.0000
22:09:25 INFO: Epoch 2 Step 2262 contrast_loss=4.4330, recon_loss=0.0000
22:09:25 INFO: Epoch 2 Step 2263 contrast_loss=4.4664, recon_loss=0.0000
22:09:25 INFO: Epoch 2 Step 2264 contrast_loss=4.4284, recon_loss=0.0000
22:09:25 INFO: Epoch 2 Step 2265 contrast_loss=4.4591, recon_loss=0.0000
22:09:25 INFO: Epoch 2 Step 2266 contrast_loss=4.4996, recon_loss=0.0000
22:09:25 INFO: Epoch 2 Step 2267 contrast_loss=4.4768, recon_loss=0.0000
22:09:25 INFO: Epoch 2 Step 2268 contrast_loss=4.3667, recon_loss=0.0000
22:09:25 INFO: Epoch 2 Step 2269 contrast_loss=4.3986, recon_loss=0.0000
22:09:25 INFO: Epoch 2 Step 2270 contrast_loss=4.4488, recon_loss=0.0000
22:09:25 INFO: Epoch 2 Step 2271 contrast_loss=4.45

22:09:29 INFO: Epoch 2 Step 2371 contrast_loss=4.3846, recon_loss=0.0000
22:09:29 INFO: Epoch 2 Step 2372 contrast_loss=4.3867, recon_loss=0.0000
22:09:29 INFO: Epoch 2 Step 2373 contrast_loss=4.3990, recon_loss=0.0000
22:09:29 INFO: Epoch 2 Step 2374 contrast_loss=4.3678, recon_loss=0.0000
22:09:29 INFO: Epoch 2 Step 2375 contrast_loss=4.4371, recon_loss=0.0000
22:09:29 INFO: Epoch 2 Step 2376 contrast_loss=4.3715, recon_loss=0.0000
22:09:29 INFO: Epoch 2 Step 2377 contrast_loss=4.4422, recon_loss=0.0000
22:09:29 INFO: Epoch 2 Step 2378 contrast_loss=4.4342, recon_loss=0.0000
22:09:29 INFO: Epoch 2 Step 2379 contrast_loss=4.3700, recon_loss=0.0000
22:09:29 INFO: Epoch 2 Step 2380 contrast_loss=4.3943, recon_loss=0.0000
22:09:29 INFO: Epoch 2 Step 2381 contrast_loss=4.3975, recon_loss=0.0000
22:09:29 INFO: Epoch 2 Step 2382 contrast_loss=4.4196, recon_loss=0.0000
22:09:29 INFO: Epoch 2 Step 2383 contrast_loss=4.3413, recon_loss=0.0000
22:09:29 INFO: Epoch 2 Step 2384 contrast_loss=4.43

22:09:33 INFO: Epoch 2 Step 2484 contrast_loss=4.4245, recon_loss=0.0000
22:09:33 INFO: Epoch 2 Step 2485 contrast_loss=4.3687, recon_loss=0.0000
22:09:34 INFO: Epoch 2 Step 2486 contrast_loss=4.4154, recon_loss=0.0000
22:09:34 INFO: Epoch 2 Step 2487 contrast_loss=4.3925, recon_loss=0.0000
22:09:34 INFO: Epoch 2 Step 2488 contrast_loss=4.4105, recon_loss=0.0000
22:09:34 INFO: Epoch 2 Step 2489 contrast_loss=4.4726, recon_loss=0.0000
22:09:34 INFO: Epoch 2 Step 2490 contrast_loss=4.3525, recon_loss=0.0000
22:09:34 INFO: Epoch 2 Step 2491 contrast_loss=4.4070, recon_loss=0.0000
22:09:34 INFO: Epoch 2 Step 2492 contrast_loss=4.4333, recon_loss=0.0000
22:09:34 INFO: Epoch 2 Step 2493 contrast_loss=4.4166, recon_loss=0.0000
22:09:34 INFO: Epoch 2 Step 2494 contrast_loss=4.3997, recon_loss=0.0000
22:09:34 INFO: Epoch 2 Step 2495 contrast_loss=4.4030, recon_loss=0.0000
22:09:34 INFO: Epoch 2 Step 2496 contrast_loss=4.4020, recon_loss=0.0000
22:09:34 INFO: Epoch 2 Step 2497 contrast_loss=4.42